# Beginner Tutorial 8: Container Orchestration with Kubernetes

## What You Will Learn

- What Kubernetes is and what problem it solves
- Pods, nodes, namespaces, and operators
- How the Dask Kubernetes Operator works
- Resource requests vs. limits
- When K8s is appropriate vs. overkill

## Prerequisites

- Completed notebooks 01–03, 05
- Conceptual understanding of containers
- NO Kubernetes cluster required (conceptual + configuration examples)

In [ ]:
import os
import tempfile

project_dir = tempfile.mkdtemp(prefix="scalable-beginner-08-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

## 💡 Key Concept: What is Kubernetes?

**Kubernetes** (K8s) = platform for managing containers at scale.

It handles:
- **WHERE** containers run (scheduling across machines)
- **HOW MANY** containers run (scaling)
- **HEALTH** — restarts crashed containers (self-healing)
- **NETWORKING** between containers (service discovery)

**Analogy:** Air traffic controller for containers. You don't tell each
plane which runway to use — the controller optimally assigns resources.

## 💡 Key Concept: Pod

A **pod** = smallest deployable unit in K8s (usually = one container).
Each Dask worker runs in its own pod.

## 💡 Key Concept: Node

A **node** = physical or virtual machine in the cluster.
Pods are scheduled onto nodes based on available resources.

## 💡 Key Concept: Namespace

A **namespace** = isolation boundary. Different teams use different namespaces
so they can't accidentally interfere with each other.

## 💡 Key Concept: Operator

An **operator** = Kubernetes extension that knows how to manage a complex
application. The Dask Operator knows how to create/scale/manage Dask clusters.

In [ ]:
# Example: Kubernetes target in a Scalable manifest
k8s_manifest = """\
version: 1
project:
  name: energy-forecast

targets:
  local:
    provider: local
    max_workers: 4
    processes: false
    containers: none

  k8s:
    provider: kubernetes
    namespace: team-climate        # Your team's isolation boundary
    image: ghcr.io/my-org/model:v1.0  # Container with your code
    adaptive:
      minimum: 2                   # Always at least 2 workers
      maximum: 20                  # Scale up to 20 when busy
    resources:
      requests:                    # Minimum guaranteed resources
        cpu: "4"
        memory: "16Gi"
      limits:                      # Maximum allowed resources
        cpu: "4"
        memory: "16Gi"

components:
  simulation:
    cpus: 4
    memory: 16G

tasks:
  run_simulation:
    component: simulation
"""

with open("scalable.yaml", "w") as f:
    f.write(k8s_manifest)

print("Kubernetes manifest written.")
print("\nKey K8s settings:")
print("  namespace: team-climate → isolation boundary")
print("  image: ... → container with your code")
print("  adaptive: min=2, max=20 → auto-scale workers")
print("  resources.requests → minimum guaranteed CPU/memory")
print("  resources.limits → maximum allowed (prevents runaway)")

In [ ]:
from scalable.manifest import parse_manifest, validate_manifest

manifest = parse_manifest("./scalable.yaml")
errors = validate_manifest(manifest)

if not errors:
    print("✓ Kubernetes manifest is valid")
    print(f"\nTargets: {list(manifest['targets'].keys())}")
    print(f"\n💡 Same Python code works with both targets:")
    print(f"   Development: scalable run --target local")
    print(f"   Production:  scalable run --target k8s")
else:
    for err in errors:
        print(f"  ✗ {err}")

## 💡 When to Use Kubernetes

| ✅ Good fit | ❌ Overkill |
|------------|-------------|
| Team sharing a cluster | Single user on laptop |
| Need resource isolation | Simple batch job |
| Auto-scaling required | Fixed known workload |
| Already have K8s infra | No existing K8s |

K8s adds complexity. For many workflows, local + cloud Fargate is simpler.
K8s shines when you have shared infrastructure.

## 📖 Vocabulary Summary

| Term | Definition |
|------|------------|
| Container Orchestration | Automating deployment/scaling of containers |
| Kubernetes (K8s) | Industry-standard container orchestration |
| Pod | Smallest deployable unit (≈ one container) |
| Node | Machine in the K8s cluster |
| Namespace | Isolation boundary for resources |
| Operator | K8s extension managing complex apps |
| kubectl | CLI tool for Kubernetes |
| Helm | Package manager for K8s |
| Resource Requests | Minimum guaranteed CPU/memory |
| Resource Limits | Maximum allowed CPU/memory |

In [ ]:
# Cleanup
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print(f"Cleaned up {project_dir}")